In [31]:
import matplotlib.pyplot as plt
import numpy as np

# I'm using seaborn for it's fantastic default styles
import seaborn as sns
sns.set_style("whitegrid")

%matplotlib inline
%load_ext autoreload
%autoreload 2

from tutils import BaseStateSystem

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
def laplacian1D(a, dx):
    return (
        - 2 * a
        + np.roll(a,1,axis=0) 
        + np.roll(a,-1,axis=0)
    ) / (dx ** 2)

def laplacian2D(a, dx):
    return (
        - 4 * a
        + np.roll(a,1,axis=0) 
        + np.roll(a,-1,axis=0)
        + np.roll(a,+1,axis=1)
        + np.roll(a,-1,axis=1)
    ) / (dx ** 2)


In [33]:
class OneDimensionalDiffusionEquation(BaseStateSystem):
    def __init__(self, D):
        self.D = D
        self.width = 1000
        self.dx = 10 / self.width
        self.dt = 0.9 * (self.dx ** 2) / (2 * D)
        self.steps = int(0.1 / self.dt)
        
    def initialise(self):
        self.t = 0
        self.X = np.linspace(-5,5,self.width)
        self.a = np.exp(-self.X**2)
        
    def update(self):
        for _ in range(self.steps):
            self.t += self.dt
            self._update()

    def _update(self):      
        La = laplacian1D(self.a, self.dx)
        delta_a = self.dt * (self.D * La)       
        self.a += delta_a
        
    def draw(self, ax):
        ax.clear()
        ax.plot(self.X,self.a, color="r")
        ax.set_ylim(0,1)
        ax.set_xlim(-5,5)
        ax.set_title("t = {:.2f}".format(self.t))
    
one_d_diffusion = OneDimensionalDiffusionEquation(D=1)

one_d_diffusion.plot_time_evolution("diffusion.gif")

In [34]:
class ReactionEquation(BaseStateSystem):
    def __init__(self, Ra, Rb):
        self.Ra = Ra
        self.Rb = Rb
        self.dt = 0.02
        self.steps = int(0.48 / self.dt)
        
    def initialise(self):
        self.t = 0
        self.a = 0.1
        self.b = 0.7
        self.Ya = []
        self.Yb = []
        self.X = []
        
    def update(self):
        for _ in range(self.steps):
            self.t += self.dt
            self._update()

    def _update(self):      
        delta_a = self.dt * self.Ra(self.a,self.b)      
        delta_b = self.dt * self.Rb(self.a,self.b)      

        self.a += delta_a
        self.b += delta_b
        
    def draw(self, ax):
        ax.clear()
        
        self.X.append(self.t)
        self.Ya.append(self.a)
        self.Yb.append(self.b)

        ax.plot(self.X,self.Ya, color="r", label="u")
        ax.plot(self.X,self.Yb, color="b", label="v")
        ax.legend()
        
        ax.set_ylim(0,1)
        ax.set_xlim(0,24)
        ax.set_xlabel("t")
        ax.set_ylabel("Concentrations")
        
alpha,beta =  0.425, 0.075

def Ra_Schnakenberg(u,v): return alpha-u+u**2*v
def Rb_Schnakenberg(u,v): return beta-u**2*v
    
one_d_reaction = ReactionEquation(Ra_Schnakenberg, Rb_Schnakenberg)
one_d_reaction.plot_time_evolution("reaction_schnakenberg.gif", n_steps=60)

In [36]:
F,k=0.75/7,0.15-0.75/7
def Ra_gray_scott(u,v): return F*(1-u)-u*v**2
def Rb_gray_scott(u,v): return u*v**2-(F+k)*v
one_d_reaction = ReactionEquation(Ra_gray_scott, Rb_gray_scott)
one_d_reaction.plot_time_evolution("reaction_gray_scott.gif", n_steps=60)